# Objetivo -
Desenvolver e comparar modelos de classificação para prever a sobrevivência de passageiros do Titanic utilizando técnicas de preparação, validação e otimização. O modelo final deverá ser utilizado para gerar previsões para o conjunto 'test.csv' e submetido ao Kaggle para avaliação.

# Bibliotecas -

In [390]:
# Bibliotecas básicas
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Divisão e validação dos dados
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score

# Pré-processamento
from sklearn.preprocessing import StandardScaler

# Balanceamento
from imblearn.over_sampling import SMOTE

# Modelos
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# Métricas
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

In [391]:
# Paleta de cores do projeto
VERMELHO_ESCURO = '#8B1E1E'
VERMELHO = '#B23A3A'
VINHO = '#5C1515'
PRETO = '#1C1C1C'
CINZA = "#727272"
BRANCO = '#FFFFFF'

# 1. Preparação inicial dos dados

In [392]:
# Carregando o dataset
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

In [393]:
# Preparando os dados
X = train.drop('Survived', axis=1)
y = train['Survived']

print(X.shape)
print(y.shape)

(891, 11)
(891,)


In [394]:
# Informações sobre o dataset
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Pclass       891 non-null    int64  
 2   Name         891 non-null    object 
 3   Sex          891 non-null    object 
 4   Age          714 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Ticket       891 non-null    object 
 8   Fare         891 non-null    float64
 9   Cabin        204 non-null    object 
 10  Embarked     889 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 76.7+ KB


In [395]:
X.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [396]:
# Removendo colunas
X = X.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

In [397]:
X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S


# 2. Tratamento de valores ausentes

In [398]:
X.isnull().sum()

Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [399]:
# Com apenas 2 registros faltantes, podemos preencher com a moda (valor mais frequente) da coluna Embarked.
X['Embarked'] = X['Embarked'].fillna(X['Embarked'].mode()[0])

In [400]:
# Com 177 registros faltantes, podemos preencher com a mediana da coluna Age.
X['Age'] = X['Age'].fillna(X['Age'].median())

In [401]:
X.isnull().sum()

Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

# 3. Tratamento de variáveis categóricas

In [402]:
print(X['Sex'].unique())
print(X['Embarked'].unique())

['male' 'female']
['S' 'C' 'Q']


In [403]:
X['Sex'] = X['Sex'].map({'male': 0, 'female': 1})

X['Embarked'] = X['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

In [404]:
X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,0,22.0,1,0,7.2500,0
1,1,1,38.0,1,0,71.2833,1
2,3,1,26.0,0,0,7.9250,0
3,1,1,35.0,1,0,53.1000,0
4,3,0,35.0,0,0,8.0500,0


# 4. Divisão dos dados para validação

In [405]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [406]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(712, 7)
(179, 7)
(712,)
(179,)


# 5. Baseline

In [407]:
# SVM
svm_model = SVC(random_state=42)

svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

In [408]:
print('Acurácia:', accuracy_score(y_test, y_pred_svm))

Acurácia: 0.6201117318435754


Baseline SVM: 62,01% de acurácia no conjunto de validação.

In [409]:
# XGBoost
xgb_model = xgb.XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

In [410]:
print('Acurácia:', accuracy_score(y_test, y_pred_xgb))

Acurácia: 0.8044692737430168


Baseline XGBoost: 80% de acurácia no conjunto de validação.

# 6. Pré-processamento/Balanceamento

In [411]:
# StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [412]:
# SVM com dados escalonados
svm_scaled = SVC(random_state=42)

svm_scaled.fit(X_train_scaled, y_train)

y_pred_svm_scaled = svm_scaled.predict(X_test_scaled)

In [413]:
print('Acurácia:', accuracy_score(y_test, y_pred_svm_scaled))

Acurácia: 0.8156424581005587


A aplicação do StandardScaler melhorou significativamente o desempenho do SVM, elevando sua acurácia de 62,01% para 81,56%. Isso demonstra a importância da padronização para esse modelo.

In [414]:
# Balanceamento com SMOTE
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

In [415]:
print(y_train_smote.value_counts())

Survived
1    439
0    439
Name: count, dtype: int64


In [416]:
# SVM com dados escalonados e balanceados
svm_smote = SVC(random_state=42)

svm_smote.fit(X_train_smote, y_train_smote)

y_pred_svm_smote = svm_smote.predict(X_test_scaled)

In [417]:
print('Acurácia:', accuracy_score(y_test, y_pred_svm_smote))

Acurácia: 0.7988826815642458


A aplicação do SMOTE ao conjunto de treinamento reduziu a acurácia do SVM de 81,56% para 79,89%, indicando que o balanceamento das classes não proporcionou melhoria no desempenho nesta configuração.

# 7. Duelo

In [418]:
resultados_duelo = pd.DataFrame({
    'Modelo': [
        'SVM - Baseline',
        'XGBoost - Baseline',
        'SVM + StandardScaler',
        'SVM + StandardScaler + SMOTE'
    ],
    'Acurácia': [
        accuracy_score(y_test, y_pred_svm),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_svm_scaled),
        accuracy_score(y_test, y_pred_svm_smote)
    ]
})

resultados_duelo

,Modelo,Acurácia
0,SVM - Baseline,0.620112
1,XGBoost - Baseline,0.804469
2,SVM + StandardScaler,0.815642
3,SVM + StandardScaler + SMOTE,0.798883


# 8. Otimização e Cross Validation

In [419]:
# Grid Search para SVM
param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

In [420]:
grid_svm = GridSearchCV(
    SVC(random_state=42),
    param_grid_svm,
    cv=5,
    scoring='accuracy'
)

grid_svm.fit(X_train_scaled, y_train)

,estimator,SVC(random_state=42)
,param_grid,"{'C': [0.1, 1, ...], 'gamma': ['scale', 'auto'], 'kernel': ['linear', 'rbf']}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,C,1


In [421]:
print(grid_svm.best_params_)
print(grid_svm.best_score_)

{'C': 1, 'gamma': 'auto', 'kernel': 'rbf'}
0.8286910272825766


In [422]:
# Melhor modelo encontrado
svm_otimizado = grid_svm.best_estimator_

# Previsões
y_pred_svm_otimizado = svm_otimizado.predict(X_test_scaled)

# Acurácia
print('Acurácia:', accuracy_score(y_test, y_pred_svm_otimizado))

Acurácia: 0.8156424581005587


In [423]:
# Grid de hiperparâmetros do XGBoost
param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.1]
}

grid_xgb = GridSearchCV(
    xgb.XGBClassifier(
        random_state=42,
        eval_metric='logloss'
    ),
    param_grid_xgb,
    cv=5,
    scoring='accuracy'
)

grid_xgb.fit(X_train, y_train)

print('Melhores parâmetros:', grid_xgb.best_params_)
print('Melhor CV:', grid_xgb.best_score_)

Melhores parâmetros: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
Melhor CV: 0.8273416724120949


In [424]:
# Melhor modelo encontrado
xgb_otimizado = grid_xgb.best_estimator_

# Previsões
y_pred_xgb_otimizado = xgb_otimizado.predict(X_test)

# Acurácia
print('Acurácia:', accuracy_score(y_test, y_pred_xgb_otimizado))

Acurácia: 0.7988826815642458


A otimização dos hiperparâmetros apresentou resultados próximos entre os dois modelos. O SVM otimizado alcançou 82,87% de acurácia média no Cross Validation e 81,56% no conjunto de teste, enquanto o XGBoost otimizado obteve 82,73% no Cross Validation e 79,89% no teste. Assim, as configurações otimizadas não produziram melhora no conjunto de teste em relação às melhores configurações anteriores, mostrando que o ajuste dos hiperparâmetros, por si só, não garantiu aumento de desempenho.

# 9. Modelo final

In [425]:
# Modelo final
modelo_final = grid_svm.best_estimator_

print('Parâmetros:', modelo_final.get_params())

Parâmetros: {'C': 1, 'break_ties': False, 'cache_size': 200, 'class_weight': None, 'coef0': 0.0, 'decision_function_shape': 'ovr', 'degree': 3, 'gamma': 'auto', 'kernel': 'rbf', 'max_iter': -1, 'probability': False, 'random_state': 42, 'shrinking': True, 'tol': 0.001, 'verbose': False}


In [426]:
# Treinamento do modelo final
modelo_final.fit(X_train_scaled, y_train)

,C,1
,kernel,'rbf'
,degree,3
,gamma,'auto'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


# 10. Submission

In [427]:
# Preparação do teste do Kaggle
X_kaggle = test.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

X_kaggle['Age'] = X_kaggle['Age'].fillna(X['Age'].median())
X_kaggle['Fare'] = X_kaggle['Fare'].fillna(X['Fare'].median())

X_kaggle['Sex'] = X_kaggle['Sex'].map({'male': 0, 'female': 1})
X_kaggle['Embarked'] = X_kaggle['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

In [428]:
# Padronização do teste
X_kaggle_scaled = scaler.transform(X_kaggle)

In [429]:
# Previsões para o Kaggle
predicoes_kaggle = modelo_final.predict(X_kaggle_scaled)

print(predicoes_kaggle[:10])

[0 0 0 0 0 0 1 0 1 0]


In [433]:
# Criando o arquivo de submission
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predicoes_kaggle
})

submission.to_csv('../outputs/submissions/submission.csv', index=False)

submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0


# 11. Kaggle e análise de score

O duelo mostrou que o SVM teve desempenho superior ao XGBoost nas configurações testadas, especialmente após a padronização. A otimização por Cross Validation encontrou uma configuração adequada para o SVM, mas não trouxe ganho adicional no conjunto de teste. No Kaggle, o modelo alcançou 0,78229, permitindo avaliar seu desempenho em dados externos ao processo de validação realizado durante o desenvolvimento.